### Install libraries

In [2]:
!pip install -q transformers datasets accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.1 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 whic

### Import libraries

In [3]:
import torch
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments
)

### Load the EmpatheticDialogues Dataset

The EmpatheticDialogues dataset contains conversations where one speaker shares an experience and the other responds empathetically.

We'll use the training split to fine-tune DistilGPT2.

In [4]:
from datasets import load_dataset   # Load the EmpatheticDialogues dataset from Hugging Face

dataset = load_dataset(
    "facebook/empathetic_dialogues",
    revision="refs/convert/parquet"
)

default/train/0000.parquet:   0%|          | 0.00/5.86M [00:00<?, ?B/s]

default/validation/0000.parquet:   0%|          | 0.00/1.00M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/954k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [5]:
 print(dataset)    #Display information in the dataset

DatasetDict({
    train: Dataset({
        features: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags'],
        num_rows: 76673
    })
    validation: Dataset({
        features: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags'],
        num_rows: 12030
    })
    test: Dataset({
        features: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags'],
        num_rows: 10943
    })
})


In [7]:
train_dataset = dataset["train"]    #Select the train split

### Explore the Dataset

In [8]:
train_dataset[0]    # Display the first training example

{'conv_id': 'hit:0_conv:1',
 'utterance_idx': 1,
 'context': 'sentimental',
 'prompt': 'I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in the world.',
 'speaker_idx': 1,
 'utterance': 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people_comma_ we felt like the only people in the world.',
 'selfeval': '5|5|5_2|2|5',
 'tags': ''}

### Explore the dataset statistics

In [9]:
### Number of utterances

print("Total Utterances:", len(train_dataset))

# Number of unique conversations

print("Total Conversations:", len(set(train_dataset["conv_id"])))

Total Utterances: 76673
Total Conversations: 17844


### Create Conversation Groups

Each conversation consists of multiple utterances.

We first group all utterances using their conversation ID so that messages remain in chronological order.

In [10]:
from collections import defaultdict

# Group all messages using conversation IDs

conversations = defaultdict(list)

for row in train_dataset:
    conversations[row["conv_id"]].append(row)

print("Total Conversations:", len(conversations))

Total Conversations: 17844


### Inspect One Conversation

In [11]:
### Display one complete conversation

sample_conv = list(conversations.values())[0]

sample_conv = sorted(
    sample_conv,
    key=lambda x: x["utterance_idx"]
)

for message in sample_conv:
    print(f"Speaker {message['speaker_idx']}: {message['utterance']}")
    print("-"*60)

Speaker 1: I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people_comma_ we felt like the only people in the world.
------------------------------------------------------------
Speaker 0: Was this a friend you were in love with_comma_ or just a best friend?
------------------------------------------------------------
Speaker 1: This was a best friend. I miss her.
------------------------------------------------------------
Speaker 0: Where has she gone?
------------------------------------------------------------
Speaker 1: We no longer talk.
------------------------------------------------------------
Speaker 0: Oh was this something that happened because of an argument?
------------------------------------------------------------


### Prepare Training Pairs

For chatbot training, we only want the assistant's replies.

Therefore, we create training examples only when:

Speaker 1 → Speaker 0

This prevents the model from learning to behave like the user.

In [13]:
### Create user-assistant conversation pairs for supervised fine-tuning

training_examples = []

for conv_id, messages in conversations.items():

    # Sort conversation by utterance order
    messages = sorted(messages, key=lambda x: x["utterance_idx"])

    # Pair every utterance with the next utterance
    for i in range(len(messages) - 1):

        current = messages[i]
        nxt = messages[i + 1]

        training_examples.append({

            "context": current["context"],

            "input": current["utterance"],

            "response": nxt["utterance"]

        })

print("Training Examples:", len(training_examples))

Training Examples: 58829


### Inspect training examples

In [14]:
### Display the first three training examples

for i in range(3):

    print(f"\nExample {i+1}\n")

    print(training_examples[i])


Example 1

{'context': 'sentimental', 'input': 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people_comma_ we felt like the only people in the world.', 'response': 'Was this a friend you were in love with_comma_ or just a best friend?'}

Example 2

{'context': 'sentimental', 'input': 'Was this a friend you were in love with_comma_ or just a best friend?', 'response': 'This was a best friend. I miss her.'}

Example 3

{'context': 'sentimental', 'input': 'This was a best friend. I miss her.', 'response': 'Where has she gone?'}


### Convert to Hugging Face Dataset

The Hugging Face Trainer API expects data in Dataset format instead of a regular Python list.

In [15]:
from datasets import Dataset

# Convert training examples to Hugging Face Dataset

hf_dataset = Dataset.from_list(training_examples)

print(hf_dataset)

Dataset({
    features: ['context', 'input', 'response'],
    num_rows: 58829
})


### Inspect dataset

In [16]:
### Display the first dataset example

hf_dataset[0]

{'context': 'sentimental',
 'input': 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people_comma_ we felt like the only people in the world.',
 'response': 'Was this a friend you were in love with_comma_ or just a best friend?'}

### Load DistilGPT2 Tokenizer

The tokenizer converts text into numerical tokens that the model can process.

In [17]:
### Load DistilGPT2 tokenizer

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

# DistilGPT2 has no default padding token
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Tokenize the Dataset

Each training example is converted into a single conversational format:

Emotion: ...

User: ...

Assistant: ...

In [18]:
### Define the maximum token sequence length for training

MAX_LENGTH = 128

# Tokenization function

def preprocess(example):

    # Combine the conversation into a single text sequence
    text = (
        f"Emotion: {example['context']}\n"
        f"User: {example['input']}\n"
        f"Assistant: {example['response']}"
    )

    # Tokenize the sequence
    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    # Labels are identical to input IDs for causal language modeling
    encoding["labels"] = encoding["input_ids"].copy()

    return encoding

### Tokenize the training dataset

In [19]:
### Apply tokenization to the entire dataset

tokenized_dataset = hf_dataset.map(preprocess)

Map:   0%|          | 0/58829 [00:00<?, ? examples/s]

### Remove unnecessary columns

In [20]:
### Remove text columns after tokenization

tokenized_dataset = tokenized_dataset.remove_columns(
    [
        "context",
        "input",
        "response"
    ]
)

print(tokenized_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 58829
})


### Inspect Tokenized Dataset

In [21]:
### Display one tokenized example

print(tokenized_dataset[0])

{'input_ids': [10161, 9650, 25, 46908, 198, 12982, 25, 314, 3505, 1016, 284, 766, 262, 26056, 351, 616, 1266, 1545, 13, 632, 373, 262, 717, 640, 356, 1683, 3377, 640, 3436, 1978, 13, 4900, 612, 373, 257, 1256, 286, 661, 62, 785, 2611, 62, 356, 2936, 588, 262, 691, 661, 287, 262, 995, 13, 198, 48902, 25, 8920, 428, 257, 1545, 345, 547, 287, 1842, 351, 62, 785, 2611, 62, 393, 655, 257, 1266, 1545, 30, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

### Load the Pre-Trained Model

In [22]:
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

### Configure Model Training

In [23]:
training_args = TrainingArguments(

    output_dir="./results",

    num_train_epochs=3,

    per_device_train_batch_size=4,

    save_strategy="epoch",

    logging_steps=100,

    learning_rate=5e-5,

    weight_decay=0.01,

    fp16=True,

    report_to="none"
)

### Fine-tune DistilGPT2

In [25]:
from transformers import DataCollatorForLanguageModeling
# Prepare batches for causal language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [26]:
### Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [27]:
### Fine-Tune the model
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,3.065468
200,2.796087
300,2.745268
400,2.760176
500,2.699502
600,2.658949
700,2.703607
800,2.675535
900,2.680910
1000,2.615029


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=22062, training_loss=2.3997316454708515, metrics={'train_runtime': 3786.7249, 'train_samples_per_second': 46.607, 'train_steps_per_second': 5.826, 'total_flos': 5764434952716288.0, 'train_loss': 2.3997316454708515, 'epoch': 3.0})

### Save the Fine-Tuned Model

In [28]:
### Save the fine-tuned model
trainer.save_model("empathetic_chatbot")

#Save the tokenizer
tokenizer.save_pretrained("empathetic_chatbot")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('empathetic_chatbot/tokenizer_config.json',
 'empathetic_chatbot/tokenizer.json')

### Test the Fine-Tuned Model

In [29]:
model_path = "empathetic_chatbot"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Load model
model = AutoModelForCausalLM.from_pretrained(model_path)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Switch model to evaluation mode
model.eval()

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

### Generate a Sample Response

In [30]:
emotion = "sad"

user_input = "I failed my final exam."

prompt = f"""Emotion: {emotion}
User: {user_input}
Assistant:"""

In [31]:
### Tokenize the output
inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

# Move inputs to the same device as the model
inputs = {k: v.to(model.device) for k, v in inputs.items()}

In [32]:
### Generate a response
output = model.generate(
    **inputs,
    max_new_tokens=50,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id
)

In [33]:
response = tokenizer.decode(output[0], skip_special_tokens=True)

# Keep only the assistant's reply
response = response.split("Assistant:")[-1]

# Remove any continuation into the next user turn
response = response.split("User:")[0].strip()

print(response)

Oh no_comma__comma_.  Hopefully you can find a job soon!  You should be able to do it in the future. :-/  I'm sure you will. How did you overcome your depression? And what happened


In [34]:
response = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print(response)

Emotion: sad
User: I failed my final exam.
Assistant: Oh no_comma__comma_.  Hopefully you can find a job soon!  You should be able to do it in the future. :-/  I'm sure you will. How did you overcome your depression? And what happened


### Chat Function

In [36]:
import torch

def chat(user_input, emotion):

    prompt = f"""Emotion: {emotion}
User: {user_input}
Assistant:"""

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.7,
            top_k=50,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return response.split("Assistant:")[-1].strip()

### Evaluate the Chatbot on Sample Conversation

In [37]:
examples = [

    ("I failed my exam.", "sad"),

    ("I feel lonely these days.", "lonely"),

    ("My best friend moved away.", "sad"),

    ("I'm nervous about tomorrow.", "afraid"),

    ("I got my dream job!", "proud")
]

for message, emotion in examples:

    print("="*70)
    print("User:", message)
    print("Emotion:", emotion)
    print()
    print("Bot:", chat(message, emotion))
    print()

User: I failed my exam.
Emotion: sad

Bot: Oh no. Did you study hard?   Did you get a good grade?   Did you have to study hard?   I am sorry to hear that.  Did you study hard?

User: I feel lonely these days.
Emotion: lonely

Bot: Oh no! Why did you feel alone? Did you have a great time?   I'm sure you'll find someone soon!  Have you tried talking to someone to try to talk to you?  I hope so!  I'm sure you'll be fine.  You have to be

User: My best friend moved away.
Emotion: sad

Bot: Oh no_comma_ what happened?  Was she alone?  Was the dog okay?  What happened?  Did you get to see her?  That's terrible.  What happened?  Did she not give you a reason why?  I'm so sorry.  What happened

User: I'm nervous about tomorrow.
Emotion: afraid

Bot: Why are you nervous?  Are you worried about it?  I bet you're scared too!  Are you scared about it?  I've had some panic attacks.  Are you going to get it checked out?  What are you going to do?  I'm so scared!

User: I got my dream job!
Emotion: p

### Calculating perplexity

In [38]:
import math

loss = None
for entry in reversed(trainer.state.log_history):
    if "loss" in entry:
        loss = entry["loss"]
        break

if loss is not None:
    perplexity = math.exp(loss)
    print("Final Loss:", loss)
    print("Perplexity:", perplexity)
else:
    print("Could not find 'loss' in log history.")

Final Loss: 2.2789320373535156
Perplexity: 9.766244853428818
